In [216]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [217]:
import torch
torch.cuda.is_available()

True

# Generate a mock dataset

In [218]:
import random
random.seed(55)
# why do we need this part? Can we make it more efficient or make it in pandas dataframe in one go?
CATALOG = {
    1: ("SmartTerm 20", "Term"),
    2: ("Term100 Protect", "Term"),
    3: ("LegacyGold Whole Life", "WholeLife"),
    4: ("Prestige Whole Life", "WholeLife"),
    5: ("WealthBuilder Endowment", "Endowment"),
    6: ("SavingsPlan Endowment", "Endowment"),
    7: ("FlexiInvest Linked", "ILP"),
    8: ("CI Shield", "CriticalIllness"),
}

ITEM_IDS_BY_CAT = {}
for item_id, (_, cat) in CATALOG.items():
    ITEM_IDS_BY_CAT.setdefault(cat, []).append(item_id)

ITEM_IDS_BY_CAT

{'Term': [1, 2],
 'WholeLife': [3, 4],
 'Endowment': [5, 6],
 'ILP': [7],
 'CriticalIllness': [8]}

In [219]:
COVERAGE_MULTIPLE = {
    "Term": (150, 300),
    "WholeLife": (40, 80),
    "Endowment": (3, 8),
    "ILP": (10, 30),
    "CriticalIllness": (50, 100),
}

In [220]:
BASE_COVERAGE = {
    "Term": 1.5,
    "WholeLife": 1.0,
    "Endowment": 0.4,
    "ILP": 0.6,
    "CriticalIllness": 0.3
}

In [221]:
def age_factor(age:float, factor_a:int=25, factor_b:int=45, offset:float=0.8)->float:
    return 1.0 + max(0.0, (age-factor_a)/factor_b) * offset

In [222]:
def sample_category(age: float)->str:
    weights = {
        "Term": max(0.1, 1.5-age/40),
        "CriticalIllness": max(0.1, 1.2-age/50),
        "WholeLife": min(1.5, age/35),
        "Endowment": min(1.3, age/45),
        "ILP": 0.7
    }
    cats = list(weights.keys())
    probs = list(weights.values())
    return random.choices(cats, weights=probs, k=1)[0]

In [223]:
def sample_item_in_category(cat: str)->int:
    return random.choice(ITEM_IDS_BY_CAT[cat])

In [224]:
def lognormal_noise(sigma: float=0.15)->float:
    return float(torch.exp(torch.randn(1) *sigma))

In [225]:
def generate_customer(customer_id: int, n_events: int | None = None, max_events:int = 8):
    if n_events is None:
        n_events = random.randint(2, max_events)
    wealth = float(torch.exp(torch.randn(1) * 0.5 + 2.0))
    age = float(random.randint(22, 55))
    items, ages, prices, sum_insures = [], [], [], []
    for _ in range(n_events):
        cat = sample_category(age)
        item_id = sample_item_in_category(cat)
        low, high = COVERAGE_MULTIPLE[cat]
        multiple = random.uniform(low, high)
        sum_insure = wealth * BASE_COVERAGE[cat] * 10_000 * lognormal_noise()
        price = sum_insure / (multiple * age_factor(age)) * lognormal_noise()

        items.append(item_id)
        ages.append(age)
        prices.append(price)
        sum_insures.append(sum_insure)

        age += random.randint(1, 5)
    return items, ages, prices, sum_insures

In [226]:
def generate_dataset(n_customers: int = 500):
    return [generate_customer(customer_id=i) for i in range(n_customers)]

In [227]:
items, ages, prices, sum_insures = generate_customer(customer_id=1, n_events=6)

for item_id, age, price, sum_insure in zip(items, ages, prices, sum_insures):
    name, cat = CATALOG[item_id]
    ratio = sum_insure / price
    print(f"age {age:>4.0f} | {name:<24} ({cat:<15}) | "
            f"price {price:>10,.0f} | sum_insure {sum_insure:>12,.0f} | ratio {ratio:>6.1f}x")

age   27 | FlexiInvest Linked       (ILP            ) | price      1,057 | sum_insure       16,335 | ratio   15.5x
age   30 | FlexiInvest Linked       (ILP            ) | price        982 | sum_insure       19,921 | ratio   20.3x
age   33 | FlexiInvest Linked       (ILP            ) | price      1,111 | sum_insure       23,850 | ratio   21.5x
age   37 | Prestige Whole Life      (WholeLife      ) | price        390 | sum_insure       33,765 | ratio   86.6x
age   41 | FlexiInvest Linked       (ILP            ) | price        698 | sum_insure       19,565 | ratio   28.0x
age   42 | FlexiInvest Linked       (ILP            ) | price        692 | sum_insure       20,896 | ratio   30.2x


# Construct it into pytorch dataset and dataloader

In [228]:
import torch
from torch.utils.data import Dataset, DataLoader, random_split

MAX_EVENTS = 8

def pad(seq, max_len, pad_value:int | float =0):
    return seq[:max_len] + ([pad_value] * max(0, max_len-len(seq)))

In [229]:
class TransactionDataset(Dataset):
    """Return items, ages, prices, sum_insures"""
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        items, ages, prices, sum_insures = self.samples[idx]
        return (
            torch.tensor(pad(items, MAX_EVENTS), dtype=torch.long),
            torch.tensor(pad(ages, MAX_EVENTS, 0.0), dtype=torch.float).unsqueeze(-1),
            torch.tensor(pad(prices, MAX_EVENTS, 0.0), dtype=torch.float).unsqueeze(-1),
            torch.tensor(pad(sum_insures, MAX_EVENTS, 0.0), dtype=torch.float).unsqueeze(-1),
        )

In [230]:
samples = generate_dataset(n_customers=500)
dataset = TransactionDataset(samples)

n_total = len(dataset)
n_train = int(n_total * .7)
n_val = int(n_total * .15)
n_test = n_total - n_train - n_val

train_set, val_set, test_set = random_split(
    # this approach can be used with Dataset subclass?
    dataset, [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(55)
)

len(train_set), len(val_set), len(test_set)

(350, 75, 75)

In [231]:
BATCH_SIZE = 32
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False)

In [232]:
class Normalizer:
    def __init__(self):
        self.mean:float | None = None
        self.std:float | None = None

    def fit(self, values: torch.Tensor):
        self.mean = values.mean().item()
        self.std = values.std().item()

    def transform(self, values: torch.Tensor) -> torch.Tensor:
        return (values - self.mean) / self.std

    def inverse_transform(self, values: torch.Tensor) -> torch.Tensor:
        return (values * self.std) + self.mean

In [233]:
def collect_train_values(base_dataset, train_subset, field_idx):
    values = []
    for i in train_subset.indices:
        values.extend(base_dataset.samples[i][field_idx])
    return torch.tensor(values, dtype=torch.float)

In [234]:
age_normalizer = Normalizer()
price_normalizer = Normalizer()
sum_insure_normalizer = Normalizer()

In [235]:
age_normalizer.fit(collect_train_values(dataset, train_set, field_idx=1))
price_normalizer.fit(collect_train_values(dataset, train_set, field_idx=2))
sum_insure_normalizer.fit(collect_train_values(dataset, train_set, field_idx=3))

In [236]:
items, ages, prices, sum_insures = next(iter(train_loader))
print(items.shape, ages.shape, prices.shape, sum_insures.shape)

ages_norm = age_normalizer.transform(ages)
print(f"normalized age batch mean ≈ {ages_norm.mean().item():.2f} (should be near 0, not exactly)")

torch.Size([32, 8]) torch.Size([32, 8, 1]) torch.Size([32, 8, 1]) torch.Size([32, 8, 1])
normalized age batch mean ≈ -1.53 (should be near 0, not exactly)


# Fusion design

In [237]:
import torch
import torch.nn as nn
from typing import Literal

class FusionEmbedding(nn.Module):
    def __init__(self, categorical_vocab_sizes:list[int], num_continuous: int, d_model:int, strategy:Literal['sum', 'concat_all', 'concat_emb_summed'] = "sum"):
        super().__init__()
        assert strategy in ("sum", "concat_all", "concat_emb_summed")
        self.strategy = strategy
        self.num_categorical = len(categorical_vocab_sizes)
        self.num_continuous = num_continuous
        self.categorical_embs = nn.ModuleList([
            nn.Embedding(vocab_size+1, d_model, padding_idx=0)
            for vocab_size in categorical_vocab_sizes
        ])
        self.continuous_projs = nn.ModuleList([
            nn.Linear(1, d_model)
            for _ in range(num_continuous)
        ])
        if strategy == "concat_all":
            concat_dim = d_model * (self.num_categorical + self.num_continuous)
            self.project_down = nn.Linear(concat_dim, d_model)

        elif strategy == "concat_emb_summed":
            concat_dim = d_model*2
            self.project_down = nn.Linear(concat_dim, d_model)

        else:
            self.project_down = None
        
    def forward(self, categorical_features: list[torch.Tensor], continuous_features: list[torch.Tensor]) -> torch.Tensor:
        assert len(categorical_features) == self.num_categorical
        assert len(continuous_features) == self.num_continuous

        cat_vecs = [
            emb(feat) for emb, feat in zip(self.categorical_embs, categorical_features)
        ]

        cont_vecs = [
            emb(feat) for emb, feat in zip(self.continuous_projs, continuous_features)
        ]
        
        if self.strategy == "sum":
            return sum(cat_vecs) + sum(cont_vecs)
        
        if self.strategy == "concat_all":
            fused = torch.cat(cat_vecs + cont_vecs, dim=-1)
            return self.project_down(fused)
        
        fused = torch.cat([sum(cat_vecs), sum(cont_vecs)], dim=-1)
        return self.project_down(fused)

In [238]:
batch, seq_len, d_model = 4, 8, 32
categorical_vocab_sizes = [8, 5]   # e.g. item_id (8 items), sales_channel (5 channels)
num_continuous = 3                  # age, price, sum_insure

item_id    = torch.randint(0, categorical_vocab_sizes[0] + 1, (batch, seq_len))
channel_id = torch.randint(0, categorical_vocab_sizes[1] + 1, (batch, seq_len))
age        = torch.randn(batch, seq_len, 1)
price      = torch.randn(batch, seq_len, 1)
sum_insure = torch.randn(batch, seq_len, 1)

for strategy in ["sum", "concat_all", "concat_emb_summed"]:
    fusion = FusionEmbedding(categorical_vocab_sizes, num_continuous, d_model, strategy=strategy)
    out = fusion([item_id, channel_id], [age, price, sum_insure])
    print(f"{strategy:<20} -> {out.shape}")   # expect (4, 8, 32) for every strategy

sum                  -> torch.Size([4, 8, 32])
concat_all           -> torch.Size([4, 8, 32])
concat_emb_summed    -> torch.Size([4, 8, 32])


In [239]:
test_emb = nn.Embedding(10, 10)
test_lin = nn.Linear(1, 10)
items = torch.tensor([1,2,3], dtype=torch.long)
ages = torch.tensor([25, 30, 45], dtype=torch.float).unsqueeze(-1)

In [240]:
iemb = test_emb(items)
alin = test_lin(ages)

In [241]:
iemb.shape, alin.shape

(torch.Size([3, 10]), torch.Size([3, 10]))

In [242]:
(iemb+alin).shape

torch.Size([3, 10])

In [243]:
torch.cat([iemb, alin], dim=-1).shape

torch.Size([3, 20])

# Define loss

In [244]:
import torch.nn as nn

class UncertaintyWeightedLoss(nn.Module):
    def __init__(self, task_types: list[str]):
        super().__init__()
        self.task_types = task_types
        self.log_vars = nn.Parameter(torch.zeros(len(task_types)))

    def forward(self, losses: list[torch.Tensor]) -> torch.Tensor:
        total = torch.tensor(0.0)
        for i, (loss, task_type) in enumerate(zip(losses, self.task_types)):
            precision = torch.exp(-self.log_vars[i]) 
            if task_type == "classification":
                total += (precision * loss) + self.log_vars[i]
            else:
                total += (precision * loss) + (0.5 * self.log_vars[i])
            
        return total

    def precisions(self):
        return {i: torch.exp(-self.log_vars[i]).item() for i in range(len(self.task_types))}

# Encoder Block

## Setup

In [245]:
import torch
import torch.nn as nn
from package.encoder_block import EncoderBlock

class EncoderCLSBackbone(nn.Module):
    def __init__(self, fusion, d_model: int, n_heads: int, max_len: int, num_layers: int =2):
        super().__init__()
        self.fusion = fusion
        self.cls = nn.Parameter(torch.randn(1, 1, d_model))
        self.layers = nn.ModuleList([
            # max_len must be added 1 because we will implement cls prepending
            EncoderBlock(d_model, n_heads, max_len+1) for _ in range(num_layers)
        ])
    
    def forward(self, categorical_features, continuous_features, pad_mask_source):
        batch = pad_mask_source.shape[0]
        pad_mask = pad_mask_source == 0
        cls_col = torch.zeros(batch, 1, dtype=torch.bool, device=pad_mask_source.device)
        pad_mask = torch.concat([cls_col, pad_mask], dim=1)
        x = self.fusion(categorical_features, continuous_features)
        cls = self.cls.expand(batch, -1, -1)
        x = torch.cat([cls, x], dim=1)

        for layer in self.layers:
            x = layer(x, pad_mask)
        
        # return only CLS vector at the 1st position of each sequence
        return x[:, 0]

In [246]:
batch, seq_len, d_model, n_heads = 4, 8, 32, 4
num_items, num_continuous = 8, 3

fusion = FusionEmbedding([num_items], num_continuous, d_model, strategy="sum")
backbone = EncoderCLSBackbone(fusion, d_model=d_model, n_heads=n_heads, max_len=seq_len, num_layers=2)

item_id    = torch.randint(0, num_items + 1, (batch, seq_len))
age        = torch.randn(batch, seq_len, 1)
price      = torch.randn(batch, seq_len, 1)
sum_insure = torch.randn(batch, seq_len, 1)

summary = backbone([item_id], [age, price, sum_insure], pad_mask_source=item_id)
print(summary.shape)   # expect (4, 32) — one vector per customer, actually collapsed this time

torch.Size([4, 32])


In [247]:
def expand_into_prefixes(samples):
    expanded = []
    for items, ages, prices, sum_insures in samples:
        n = len(items)
        for prefix_len in range(1, n):
            expanded.append((
                items[:prefix_len],
                ages[:prefix_len],
                prices[:prefix_len],
                sum_insures[:prefix_len],
                items[prefix_len],
                ages[prefix_len],
                prices[prefix_len],
                sum_insures[prefix_len]
            ))
    return expanded

class NextSequenceDataset(Dataset):
    def __init__(self, expanded_samples):
        self.samples = expanded_samples

    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        items, ages, prices, sum_insures, next_item, next_age, next_price, next_sum_insure = self.samples[idx]
        return (
            torch.tensor(pad(items, MAX_EVENTS), dtype=torch.long),
            torch.tensor(pad(ages, MAX_EVENTS), dtype=torch.float).unsqueeze(-1),
            torch.tensor(pad(prices, MAX_EVENTS), dtype=torch.float).unsqueeze(-1),
            torch.tensor(pad(sum_insures, MAX_EVENTS), dtype=torch.float).unsqueeze(-1),
            torch.tensor(next_item, dtype=torch.long),
            torch.tensor([next_age], dtype=torch.float),
            torch.tensor([next_price], dtype=torch.float),
            torch.tensor([next_sum_insure], dtype=torch.float),
        )

In [248]:
def collect_raw_samples(base_dataset, subset):
    return [base_dataset.samples[i] for i in subset.indices]

train_expanded = expand_into_prefixes(collect_raw_samples(dataset, train_set))
val_expanded = expand_into_prefixes(collect_raw_samples(dataset, val_set))
test_expanded = expand_into_prefixes(collect_raw_samples(dataset, test_set))

In [249]:
next_item_train_loader = DataLoader(NextSequenceDataset(train_expanded), batch_size=BATCH_SIZE, shuffle=True)
next_item_val_loader = DataLoader(NextSequenceDataset(val_expanded), batch_size=BATCH_SIZE, shuffle=False)
next_item_test_loader = DataLoader(NextSequenceDataset(test_expanded), batch_size=BATCH_SIZE, shuffle=False)

print(f"train rows: {len(train_expanded)}  val rows: {len(val_expanded)}  test rows: {len(test_expanded)}")


train rows: 1421  val rows: 276  test rows: 318


In [250]:
class MultiTaskModel(nn.Module):
    def __init__(self, backbone, d_model: int, num_items: int):
        super().__init__()
        self.backbone = backbone
        # num_items + 1 because we reserve 0 for padding
        # item_head will return as logit
        self.item_head = nn.Linear(d_model, num_items+1)
        self.age_head = nn.Linear(d_model, 1)
        self.price_head = nn.Linear(d_model, 1)
        self.sum_insure_head = nn.Linear(d_model, 1)

    def forward(self, categorical_features, continuous_features, pad_mask_source):
        summary = self.backbone(categorical_features, continuous_features, pad_mask_source)
        # is it possible to factor this out into .predict_age_with_constraint?
        ages_norm = continuous_features[0]
        lengths = (pad_mask_source != 0).sum(dim=1)
        batch_idx = torch.arange(ages_norm.shape[0], device=ages_norm.device)
        current_age = ages_norm[batch_idx, lengths-1, 0]
        delta_age = nn.functional.softplus(self.age_head(summary).squeeze(-1))
        age_pred = (current_age + delta_age).unsqueeze(-1)
        return (
            self.item_head(summary),
            age_pred,
            self.price_head(summary),
            self.sum_insure_head(summary),
        )

In [251]:
fusion   = FusionEmbedding([num_items], num_continuous=3, d_model=32, strategy="sum")
backbone = EncoderCLSBackbone(fusion, d_model=32, n_heads=4, max_len=MAX_EVENTS, num_layers=2)
model    = MultiTaskModel(backbone, d_model=32, num_items=num_items)

items, ages, prices, sum_insures, next_item, next_age, next_price, next_sum_insure = next(iter(next_item_train_loader))

# normalize every continuous value — both the input sequence AND the regression
# labels — using the SAME normalizers already fit on the train split only
ages_norm             = age_normalizer.transform(ages)
prices_norm           = price_normalizer.transform(prices)
sum_insures_norm      = sum_insure_normalizer.transform(sum_insures)
next_age_norm         = age_normalizer.transform(next_age)
next_price_norm       = price_normalizer.transform(next_price)
next_sum_insure_norm  = sum_insure_normalizer.transform(next_sum_insure)

item_logits, age_pred, price_pred, sum_insure_pred = model(
    [items], [ages_norm, prices_norm, sum_insures_norm], pad_mask_source=items
)

loss_item       = nn.functional.cross_entropy(item_logits, next_item)
loss_age        = nn.functional.mse_loss(age_pred, next_age_norm)
loss_price      = nn.functional.mse_loss(price_pred, next_price_norm)
loss_sum_insure = nn.functional.mse_loss(sum_insure_pred, next_sum_insure_norm)

print(loss_item.item(), loss_age.item(), loss_price.item(), loss_sum_insure.item())


2.5496160984039307 0.12969359755516052 1.2433075904846191 0.9107673168182373


In [252]:
loss_weigher = UncertaintyWeightedLoss(task_types=["classification", "regression", "regression", "regression"])

total_loss = loss_weigher([loss_item, loss_age, loss_price, loss_sum_insure])
print(total_loss.item())

4.8333845138549805


In [253]:
items.shape, item_logits.shape

(torch.Size([32, 8]), torch.Size([32, 9]))

In [254]:
items[0], (item_logits[0])

(tensor([7, 0, 0, 0, 0, 0, 0, 0]),
 tensor([ 0.8605, -0.3061, -1.2003,  0.5244, -0.2043,  1.3232, -0.3936,  0.2007,
          0.0249], grad_fn=<SelectBackward0>))

In [255]:
next_item[0], torch.argmax(item_logits[0])

(tensor(3), tensor(5))

In [256]:
ages[0].squeeze(-1), age_pred[0], age_normalizer.inverse_transform(age_pred[0]), next_age[0]

(tensor([45.,  0.,  0.,  0.,  0.,  0.,  0.,  0.]),
 tensor([0.4755], grad_fn=<SelectBackward0>),
 tensor([51.0196], grad_fn=<AddBackward0>),
 tensor([50.]))

## Test train loop

In [257]:
D_MODEL = 32
N_HEADS = 4
NUM_LAYERS = 2

fusion = FusionEmbedding([num_items], num_continuous=3, d_model=D_MODEL, strategy="sum")
backbone = EncoderCLSBackbone(fusion, d_model=D_MODEL, n_heads=N_HEADS, max_len=MAX_EVENTS, num_layers=NUM_LAYERS)
model = MultiTaskModel(backbone, d_model=D_MODEL, num_items=num_items)
loss_weigher = UncertaintyWeightedLoss(task_types=["classification", "regression", "regression", "regression"])

optimizer = torch.optim.Adam(list(model.parameters())+list(loss_weigher.parameters()), lr=1e-3)
EPOCHS = 50

def normalize_batch(ages, prices, sum_insures, next_age, next_price, next_sum_insure):
    return(
        age_normalizer.transform(ages),
        price_normalizer.transform(prices),
        sum_insure_normalizer.transform(sum_insures),
        age_normalizer.transform(next_age),
        price_normalizer.transform(next_price),
        sum_insure_normalizer.transform(next_sum_insure),
    )

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    for items, ages, prices, sum_insures, next_item, next_age, next_price, next_sum_insure in next_item_train_loader:
        ages_norm, prices_norm, sum_insures_norm, next_age_norm, next_price_norm, next_sum_insure_norm = (
            normalize_batch(ages, prices, sum_insures, next_age, next_price, next_sum_insure)
        )
        item_logits, age_pred, price_pred, sum_insure_pred = model(
            [items], [ages_norm, prices_norm, sum_insures_norm], pad_mask_source=items
        )

        loss_item = nn.functional.cross_entropy(item_logits, next_item)
        loss_age = nn.functional.mse_loss(age_pred, next_age_norm)
        loss_price = nn.functional.mse_loss(price_pred, next_price_norm)
        loss_sum_insure = nn.functional.mse_loss(sum_insure_pred, next_sum_insure_norm)

        loss = loss_weigher([loss_item, loss_age, loss_price, loss_sum_insure])

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    if (epoch+1) % 5 == 0:
        weights = torch.exp(-loss_weigher.log_vars).detach()
        print(
            f"epoch {epoch+1:3d} | train loss {total_loss / len(next_item_train_loader):4f} "
            f"| weights item={weights[0]:.2f} age={weights[1]:.2f} price={weights[2]:.2f} sum_insure={weights[3]:.2f}"
        )

        model.eval()
        correct, total = 0, 0
        age_abs_err, price_abs_err, sum_insure_abs_err = 0.0, 0.0, 0.0

        with torch.no_grad():
            for items, ages, prices, sum_insures, next_item, next_age, next_price, next_sum_insure in next_item_val_loader:
                ages_norm, prices_norm, sum_insures_norm, *_ = (
                    normalize_batch(ages, prices, sum_insures, next_age, next_price, next_sum_insure)
                )

                item_logits, age_pred, price_pred, sum_insure_pred = model(
                    [items], [ages_norm, prices_norm, sum_insures_norm], pad_mask_source=items
                )

                correct += (item_logits.argmax(dim=-1) == next_item).sum().item()
                total += next_item.numel()

                age_abs_err += (age_normalizer.inverse_transform(age_pred) - next_age).abs().sum().item()
                price_abs_err += (price_normalizer.inverse_transform(price_pred) - next_price).abs().sum().item()
                sum_insure_abs_err += (sum_insure_normalizer.inverse_transform(sum_insure_pred) - next_sum_insure).abs().sum().item()
        n_val = len(next_item_val_loader.dataset)
        print(
            f"    |- val item_acc {correct/total:.2%} "
            f"| age MAE {age_abs_err/n_val:.2f} | price MAE {price_abs_err/n_val:.2f} | sum_insure MAE {sum_insure_abs_err/n_val:.2f}"
        )
        model.train()
    

epoch   5 | train loss 3.173310 | weights item=0.82 age=1.25 price=0.90 sum_insure=0.94
    |- val item_acc 18.84% | age MAE 1.26 | price MAE 1763.24 | sum_insure MAE 34235.38
epoch  10 | train loss 2.811482 | weights item=0.71 age=1.56 price=0.84 sum_insure=0.93
    |- val item_acc 16.67% | age MAE 1.26 | price MAE 1658.09 | sum_insure MAE 31578.92
epoch  15 | train loss 2.542795 | weights item=0.63 age=1.95 price=0.80 sum_insure=0.94
    |- val item_acc 15.94% | age MAE 1.29 | price MAE 1616.87 | sum_insure MAE 31733.16
epoch  20 | train loss 2.312265 | weights item=0.59 age=2.43 price=0.79 sum_insure=0.98
    |- val item_acc 15.94% | age MAE 1.29 | price MAE 1719.52 | sum_insure MAE 32534.95
epoch  25 | train loss 2.106110 | weights item=0.56 age=3.03 price=0.81 sum_insure=1.04
    |- val item_acc 18.84% | age MAE 1.28 | price MAE 1858.42 | sum_insure MAE 35107.93
epoch  30 | train loss 1.876369 | weights item=0.56 age=3.76 price=0.83 sum_insure=1.13
    |- val item_acc 17.39% | age

In [258]:
def theoretical_item_ceiling(age: float) -> float:
    """Bayes-optimal probability of guessing the exact next item correctly,
    using the generator's own category weights — the best any model can do."""
    weights = {
        "Term":            max(0.1, 1.5 - age / 40),
        "CriticalIllness": max(0.1, 1.2 - age / 50),
        "WholeLife":       min(1.5, age / 35),
        "Endowment":       min(1.3, age / 45),
        "ILP":             0.7,
    }
    total = sum(weights.values())
    best_cat, best_cat_weight = max(weights.items(), key=lambda kv: kv[1])

    cat_prob            = best_cat_weight / total                    # P(best category | age)
    item_prob_given_cat = 1 / len(ITEM_IDS_BY_CAT[best_cat])          # uniform pick within that category
    return cat_prob * item_prob_given_cat


# average over the actual ages seen in validation — each val_expanded row is
# (items, ages, prices, sum_insures, next_item, next_age, next_price, next_sum_insure)
current_ages = [ages[-1] for (items, ages, prices, sum_insures, *_ ) in val_expanded]
ceiling = sum(theoretical_item_ceiling(age) for age in current_ages) / len(current_ages)

print(f"theoretical best possible next-item accuracy: {ceiling:.1%}")

theoretical best possible next-item accuracy: 16.8%


# Decoder Block